# Figure S6: Module 1 latent-state analysis

- Fig.S6a-e
- Inputs: Module 1 PBMC latent artifacts and scGen PBMC prediction payload.
- Outputs: artifacts/paper_figures/supp/FigS6_Module1LatentState/.
- Role: PBMC latent state structure and target-domain control context.



> Refreshed figure-plan entry. This notebook preserves the existing compact plotting style and should read long-format sources from `artifacts/analysis` after server results are recovered.


In [ ]:
from __future__ import annotations

import json
import pickle
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import scanpy as sc
from IPython.display import Image, display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'scripts').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from scripts.common.paper_plot_style import apply_gears_paper_style, style_axis
from scripts.trishift.analysis.stage1_latent_clustering import run_stage1_latent_clustering

apply_gears_paper_style(font_scale=1.0)


In [ ]:
MODE = 'pbmc_celltype'  # fixed for the paper supplement
DATASET_NAME = 'scgen_pbmc'
SPLIT_ID = 1
STAGE1_POOL_MODE = 'train_all_cells'
RANDOM_SEED = 24
PBMC_SOURCE = 'scgen'
TRISHIFT_SCGEN_PKL = repo_root / 'artifacts' / 'results' / 'scgen_pbmc_celltype' / 'trishift_scgen_pbmc_celltype_1.pkl'

SOURCE_ROOT = repo_root / 'artifacts' / 'stage1_latent_clustering' / MODE / DATASET_NAME / f'split{SPLIT_ID}' / f'{STAGE1_POOL_MODE}_seed{RANDOM_SEED}'
OUT_ROOT = repo_root / 'artifacts' / 'paper_figures' / 'supp' / 'FigS6_Module1LatentState'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Preferred source:', SOURCE_ROOT)
print('Output root:', OUT_ROOT)
print('PBMC source:', PBMC_SOURCE)
print('TriShift scGen pkl:', TRISHIFT_SCGEN_PKL)


In [ ]:
if SOURCE_ROOT.exists() and (SOURCE_ROOT / 'cluster_metrics.csv').exists():
    source_dir = SOURCE_ROOT
else:
    result = run_stage1_latent_clustering(
        mode=MODE,
        dataset_name=DATASET_NAME,
        split_id=SPLIT_ID,
        module1_pool_mode=STAGE1_POOL_MODE,
        random_seed=RANDOM_SEED,
        pbmc_source=PBMC_SOURCE,
        pbmc_filter_gene_by_counts=0,
        pbmc_normalize_total=0,
        pbmc_log1p=False,
        pbmc_n_hvg=0,
    )
    source_dir = Path(result.out_dir)

metrics_df = pd.read_csv(source_dir / 'cluster_metrics.csv')
run_meta = json.loads((source_dir / 'run_meta.json').read_text(encoding='utf-8'))
metrics_df.to_csv(OUT_ROOT / 'figs6_cluster_metrics.csv', index=False, encoding='utf-8-sig')
(OUT_ROOT / 'figs6_source_run_meta.json').write_text(json.dumps(run_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print('Resolved source:', source_dir)
display(metrics_df)


In [ ]:
latent_h5ad_path = source_dir / 'latent_with_clusters.h5ad'
latent_adata = sc.read_h5ad(latent_h5ad_path) if latent_h5ad_path.exists() else None

# Re-render figure panels instead of copying old source PNGs so layout can be tuned here.
if latent_adata is not None:
    apply_gears_paper_style(font_scale=1.0)

    def _cluster_sort_key(value: str):
        text = str(value)
        return (0, int(text)) if text.isdigit() else (1, text)

    # FigS6a: UMAP by cluster
    cluster_levels = sorted(pd.Categorical(latent_adata.obs['leiden']).categories.tolist(), key=_cluster_sort_key)
    latent_adata.obs['leiden'] = pd.Categorical(latent_adata.obs['leiden'].astype(str), categories=cluster_levels, ordered=True)
    cluster_palette = sns.color_palette('tab20', n_colors=max(len(cluster_levels), 3)).as_hex()[:len(cluster_levels)]
    fig = sc.pl.umap(
        latent_adata,
        color=['leiden'],
        title=[''],
        frameon=False,
        return_fig=True,
        show=False,
        legend_loc='right margin',
        legend_fontsize=10,
        legend_fontoutline=0,
        palette=cluster_palette,
        size=14,
    )
    fig.set_size_inches(9.2, 6.8)
    fig.suptitle('scGen PBMC Module 1 latent by cluster', y=0.98, fontsize=16, fontweight='semibold')
    for ax in fig.axes:
        if hasattr(ax, 'set_title'):
            ax.set_title('')
        style_axis(ax)
    fig.subplots_adjust(top=0.88, right=0.83)
    fig.savefig(OUT_ROOT / 'figs6a_umap_by_cluster.png', dpi=320, bbox_inches='tight')
    plt.close(fig)

    # FigS6b: UMAP by cell type
    cell_types = sorted(latent_adata.obs['label_cell_type'].astype(str).unique().tolist())
    palette_ct = sns.color_palette('tab10', n_colors=max(len(cell_types), 3)).as_hex()[:len(cell_types)]
    fig = sc.pl.umap(
        latent_adata,
        color=['label_cell_type'],
        title=[''],
        frameon=False,
        return_fig=True,
        show=False,
        legend_loc='right margin',
        legend_fontsize=8.5,
        legend_fontoutline=0,
        palette=palette_ct,
        size=14,
    )
    fig.set_size_inches(10.4, 7.0)
    fig.suptitle('scGen PBMC Module 1 latent by cell type', y=0.98, fontsize=16, fontweight='semibold')
    for ax in fig.axes:
        if hasattr(ax, 'set_title'):
            ax.set_title('')
        style_axis(ax)
    fig.subplots_adjust(top=0.88, right=0.80)
    fig.savefig(OUT_ROOT / 'figs6b_umap_by_cell_type.png', dpi=320, bbox_inches='tight')
    plt.close(fig)

    # FigS6c: cluster vs cell type heatmap
    table = pd.crosstab(latent_adata.obs['label_cell_type'].astype(str), latent_adata.obs['leiden'].astype(str))
    table = table.reindex(columns=[str(x) for x in cluster_levels], fill_value=0)
    fig_w = max(8.0, 0.48 * table.shape[1] + 4.0)
    fig_h = max(5.4, 0.5 * table.shape[0] + 2.8)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=220)
    sns.heatmap(
        table.astype(float),
        cmap='Blues',
        annot=table.astype(int),
        fmt='d',
        linewidths=0.45,
        linecolor='white',
        cbar=True,
        cbar_kws={'shrink': 0.78, 'pad': 0.02},
        ax=ax,
    )
    ax.set_xlabel('Leiden cluster')
    ax.set_ylabel('Cell type')
    ax.set_title('Cluster vs cell type', fontsize=15, fontweight='semibold', pad=10)
    ax.tick_params(axis='x', rotation=35, labelsize=10)
    ax.tick_params(axis='y', rotation=0, labelsize=10)
    ax.grid(False)
    fig.tight_layout()
    fig.savefig(OUT_ROOT / 'figs6c_cluster_vs_cell_type.png', bbox_inches='tight')
    plt.close(fig)
else:
    figure_sources = {
        'figs6a_umap_by_cluster.png': source_dir / 'umap_by_cluster.png',
        'figs6b_umap_by_cell_type.png': source_dir / 'umap_by_label_cell_type.png',
        'figs6c_cluster_vs_cell_type.png': source_dir / 'cluster_vs_label_cell_type.png',
    }
    for out_name, src_path in figure_sources.items():
        if src_path.exists():
            shutil.copy2(src_path, OUT_ROOT / out_name)

metric_row = metrics_df[metrics_df['label_key'].astype(str) == 'label_cell_type'].copy()
metric_cols = [col for col in ['ARI_cluster/label', 'NMI_cluster/label', 'ASW_label', 'avg_bio'] if col in metric_row.columns]
plot_df = metric_row[metric_cols].melt(var_name='metric', value_name='value') if not metric_row.empty and metric_cols else pd.DataFrame(columns=['metric', 'value'])
fig, ax = plt.subplots(figsize=(6.8, 4.2), dpi=220)
if plot_df.empty:
    ax.text(0.5, 0.5, 'No PBMC cell-type metric summary available', ha='center', va='center')
    ax.axis('off')
else:
    plot_df['metric_label'] = plot_df['metric'].map({
        'ARI_cluster/label': 'ARI',
        'NMI_cluster/label': 'NMI',
        'ASW_label': 'ASW',
        'avg_bio': 'Avg. bio',
    }).fillna(plot_df['metric'])
    sns.barplot(data=plot_df, x='metric_label', y='value', color='#4C78A8', ax=ax)
    ax.set_xlabel('')
    ax.set_ylabel('Score')
    ax.set_title('PBMC clustering metrics', fontsize=15, fontweight='semibold', pad=10)
    style_axis(ax, grid_axis='y')
    ax.set_ylim(0, max(0.9, float(plot_df['value'].max()) + 0.08))
    for patch in ax.patches:
        patch.set_edgecolor('black')
        patch.set_linewidth(0.5)
        h = patch.get_height()
        ax.text(patch.get_x() + patch.get_width()/2, h + 0.015, f'{h:.2f}', ha='center', va='bottom', fontsize=9)
fig.tight_layout()
fig.savefig(OUT_ROOT / 'figs6d_cluster_metrics.png', bbox_inches='tight')
plt.close(fig)
plot_df.to_csv(OUT_ROOT / 'figs6d_cluster_metrics_values.csv', index=False, encoding='utf-8-sig')


In [ ]:
# FigS6e: UMAP from the TriShift scGen unseen-control export pkl.
if not TRISHIFT_SCGEN_PKL.exists():
    raise FileNotFoundError(TRISHIFT_SCGEN_PKL)

with TRISHIFT_SCGEN_PKL.open('rb') as f:
    trishift_payload = pickle.load(f)
if not isinstance(trishift_payload, dict) or 'stimulated' not in trishift_payload:
    raise ValueError(f'Unexpected TriShift scGen payload keys: {list(trishift_payload)[:10]}')

item = trishift_payload['stimulated']
ctrl_x = np.asarray(item['Ctrl_full'], dtype=np.float32)
truth_x = np.asarray(item['Truth_full'], dtype=np.float32)
pred_x = np.asarray(item['Pred_full'], dtype=np.float32)
meta = dict(item.get('export_metadata', {}))
heldout_domains = ', '.join(map(str, meta.get('test_domain_values', []))) or 'target domain'

X = np.vstack([ctrl_x, truth_x, pred_x]).astype(np.float32, copy=False)
groups = (['Control'] * ctrl_x.shape[0]) + (['Perturbed'] * truth_x.shape[0]) + (['TriShift'] * pred_x.shape[0])
scgen_pred_adata = sc.AnnData(X)
scgen_pred_adata.obs['group'] = pd.Categorical(groups, categories=['Control', 'Perturbed', 'TriShift'], ordered=True)
scgen_pred_adata.obs['condition'] = 'stimulated'
scgen_pred_adata.obs['heldout_domain'] = heldout_domains
scgen_pred_adata.var_names = np.asarray(item.get('gene_name_full', [f'g{i}' for i in range(X.shape[1])])).astype(str)

sc.pp.pca(scgen_pred_adata, n_comps=min(40, X.shape[0] - 1, X.shape[1]), svd_solver='arpack', random_state=RANDOM_SEED)
sc.pp.neighbors(scgen_pred_adata, n_neighbors=24, n_pcs=min(30, scgen_pred_adata.obsm['X_pca'].shape[1]), random_state=RANDOM_SEED)
sc.tl.umap(scgen_pred_adata, random_state=RANDOM_SEED, min_dist=0.35)

palette = {'Control': '#BDBDBD', 'Perturbed': '#5B6770', 'TriShift': '#4DB6AC'}
fig = sc.pl.umap(
    scgen_pred_adata,
    color=['group'],
    title=[''],
    frameon=False,
    return_fig=True,
    show=False,
    palette=palette,
    size=18,
    alpha=0.9,
    legend_loc='right margin',
    legend_fontsize=10,
    legend_fontoutline=0,
)
fig.set_size_inches(8.4, 6.1)
fig.suptitle(f'TriShift scGen unseen-control prediction | {heldout_domains}', y=0.98, fontsize=14, fontweight='semibold')
for ax in fig.axes:
    if hasattr(ax, 'set_title'):
        ax.set_title('')
    style_axis(ax)
fig.subplots_adjust(top=0.88, right=0.82)
fig.savefig(OUT_ROOT / 'figs6e_trishift_scgen_unseen_umap.png', dpi=320, bbox_inches='tight')
plt.close(fig)

umap_df = pd.DataFrame(scgen_pred_adata.obsm['X_umap'], columns=['UMAP1', 'UMAP2'])
umap_df['group'] = scgen_pred_adata.obs['group'].astype(str).to_numpy()
umap_df['condition'] = scgen_pred_adata.obs['condition'].astype(str).to_numpy()
umap_df['heldout_domain'] = scgen_pred_adata.obs['heldout_domain'].astype(str).to_numpy()
umap_df.to_csv(OUT_ROOT / 'figs6e_trishift_scgen_unseen_umap_points.csv', index=False, encoding='utf-8-sig')

trishift_umap_meta = {
    'pkl_path': str(TRISHIFT_SCGEN_PKL),
    'condition': 'stimulated',
    'ctrl_cells': int(ctrl_x.shape[0]),
    'truth_cells': int(truth_x.shape[0]),
    'pred_cells': int(pred_x.shape[0]),
    'n_genes': int(X.shape[1]),
    'export_metadata': {k: (v.tolist() if hasattr(v, 'tolist') else v) for k, v in meta.items()},
}
(OUT_ROOT / 'figs6e_trishift_scgen_unseen_umap_meta.json').write_text(json.dumps(trishift_umap_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(trishift_umap_meta)


In [ ]:
for image_name in [
    'figs6a_umap_by_cluster.png',
    'figs6b_umap_by_cell_type.png',
    'figs6c_cluster_vs_cell_type.png',
    'figs6d_cluster_metrics.png',
    'figs6e_trishift_scgen_unseen_umap.png',
]:
    image_path = OUT_ROOT / image_name
    print(image_path)
    if image_path.exists():
        display(Image(filename=str(image_path), width=900))
print(OUT_ROOT)


In [ ]:
from pathlib import Path
import sys
repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from scripts.trishift.analysis.render_refresh_figures import render
out = render('figs6')
print(f'Wrote {out}')
